In [1]:
import pandas as pd

In [2]:
# Cargamos los datos

data = pd.read_csv("processed_dataset.csv", sep=",", engine="python")

In [3]:
# Limpieza de datos

data = data.dropna(subset=["clean_post", "subreddit", "sentiment"])   # Eliminar valores faltantes
data["sentiment"].value_counts()

sentiment
NEGATIVE    224762
POSITIVE     47472
Name: count, dtype: int64

In [4]:
data.head()

,created_date,created_timestamp,subreddit,title,author,author_created_utc,full_link,score,num_comments,num_crossposts,subreddit_subscribers,post,sentiment,clean_post
0,2010-02-11 19:47:22,1265910442.0,analytics,So what do you guys all do related to analytic...,xtom,1.227476e+09,https://www.reddit.com/r/analytics/comments/b0...,7.0,4.0,0.0,NaN,There's a lot of reasons to want to know all t...,NEGATIVE,there lot reason want know stuff figured id ge...
1,2010-03-04 20:17:26,1267726646.0,analytics,"Google's Invasive, non-Anonymized Ad Targeting...",xtom,1.227476e+09,https://www.reddit.com/r/analytics/comments/b9...,2.0,1.0,0.0,NaN,"I'm cross posting this from /r/cyberlaw, hopef...",NEGATIVE,im cross posting rcyberlaw hopefully guy find ...
2,2011-01-06 04:51:18,1294282278.0,analytics,"DotCed - Functional Web Analytics - Tagging, R...",dotced,1.294282e+09,https://www.reddit.com/r/analytics/comments/ew...,1.0,1.0,NaN,NaN,"DotCed,a Functional Analytics Consultant, offe...",NEGATIVE,dotceda functional analytics consultant offeri...
3,2011-01-19 11:45:30,1295430330.0,analytics,Program Details - Data Analytics Course,iqrconsulting,1.288245e+09,https://www.reddit.com/r/analytics/comments/f5...,0.0,0.0,NaN,NaN,Here is the program details of the data analyt...,NEGATIVE,program detail data analytics certification co...
4,2011-01-19 21:52:28,1295466748.0,analytics,potential job in web analytics... need to anal...,therewontberiots,1.278672e+09,https://www.reddit.com/r/analytics/comments/f5...,2.0,4.0,NaN,NaN,i decided grad school (physics) was not for me...,POSITIVE,decided grad school physic branching job marke...


#### Método basado en lexicons

In [5]:
from sklearn.model_selection import train_test_split  
from sklearn.metrics import f1_score, classification_report 
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer


# Imprimir los valores únicos en la columna 'sentiment' del DataFrame
# Esto permite verificar las categorías de sentimientos presentes en los datos
print("Valores únicos en sentiment:", data["sentiment"].unique())

# El 30% de los datos será utilizado para pruebas, y el 70% restante para entrenar el modelo
X_train, X_test, y_train, y_test = train_test_split(
    data["clean_post"],  
    data["sentiment"],   
    test_size=0.3,      
    random_state=42    
)

# Inicializar el analizador de sentimientos de VADER
analyzer = SentimentIntensityAnalyzer()


def score_sentence_vader(sentence):
    '''Función que recibe una oración y devuelve el sentimiento basado en el análisis de VADER.''' 

    sentence = str(sentence)   
    score = analyzer.polarity_scores(sentence)   # Calculamos la puntuación de sentimiento de la frase
    # VADER devuelve un diccionario con varias puntuaciones, incluyendo 'compound' que es la puntuación final
    # Si el valor 'compound' es positivo, consideramos que el sentimiento es "POSITIVE", de lo contrario, "NEGATIVE"
    return "POSITIVE" if score["compound"] > 0 else "NEGATIVE"


# Aplicamos la función score_sentence_vader a cada texto en 'X_test' para obtener las predicciones de sentimiento
y_pred_vader = [score_sentence_vader(text) for text in X_test]

# Diagnóstico de las primeras predicciones y las etiquetas reales
print("\nPrimeras 5 predicciones:", y_pred_vader[:5])
print("Primeras 5 etiquetas reales:", y_test.iloc[:5].values)

# Evaluación del rendimiento del modelo
f1_vader = f1_score(y_test, y_pred_vader, average="weighted")
print("\nF1 Score (VADER):", f1_vader)
print("\nReporte de clasificación:")
print(classification_report(y_test, y_pred_vader))

Valores únicos en sentiment: ['NEGATIVE' 'POSITIVE']

Primeras 5 predicciones: ['POSITIVE', 'POSITIVE', 'POSITIVE', 'POSITIVE', 'POSITIVE']
Primeras 5 etiquetas reales: ['NEGATIVE' 'POSITIVE' 'NEGATIVE' 'NEGATIVE' 'NEGATIVE']

F1 Score (VADER): 0.3434384405829144

Reporte de clasificación:
              precision    recall  f1-score   support

    NEGATIVE       0.95      0.21      0.35     67604
    POSITIVE       0.20      0.94      0.33     14067

    accuracy                           0.34     81671
   macro avg       0.57      0.58      0.34     81671
weighted avg       0.82      0.34      0.34     81671



#### Método basado en palabras únicas

In [6]:
from nltk.tokenize import word_tokenize

# Crear vocabularios únicos
positive_vocabulary = set()   # Conjunto vacío para las palabras positivas
negative_vocabulary = set()   # Conjunto vacío para las palabras negativas

# Iteramos sobre las reseñas positivas
for review in data[data["sentiment"] == "POSITIVE"]["clean_post"]:
    words = set(review.lower().split())   # Convertimos a minúsculas y dividimos en palabras
    positive_vocabulary.update(words)   # Actualizamos el vocabulario positivo

# Iteramos sobre las reseñas negativas
for review in data[data["sentiment"] == "NEGATIVE"]["clean_post"]:
    words = set(review.lower().split())   
    negative_vocabulary.update(words)   # Actualizamos el vocabulario negativo


# Clasificación basada en vocabularios únicos
def unique_word_based_sentiment(text):
    try:
        tokens = set(word_tokenize(text.lower()))   # Convertimos el texto a minúsculas y tokenizamos
        
        # Calculamos coincidencias
        pos_matches = len(tokens.intersection(positive_vocabulary))
        neg_matches = len(tokens.intersection(negative_vocabulary))
        
        # Devolvemos la etiqueta en el formato correcto
        return "POSITIVE" if pos_matches > neg_matches else "NEGATIVE"

    except Exception as e:
        print(f"Error de procesamiento: {e}")
        return "NEGATIVE"   # Default return value

# División de los datos
X_train, X_test, y_train, y_test = train_test_split(
    data["clean_post"],
    data["sentiment"],
    test_size=0.3,
    random_state=42
)


# Predicción con vocabularios únicos
y_pred_unique = [unique_word_based_sentiment(text) for text in X_test]

# Diagnóstico
print("\nVocabulario positivo size:", len(positive_vocabulary))
print("Vocabulario negativo size:", len(negative_vocabulary))
print("\nPrimeras 5 predicciones:", y_pred_unique[:5])
print("Primeras 5 etiquetas reales:", y_test.iloc[:5].values)

# Evaluación
f1_unique = f1_score(y_test, y_pred_unique, average="weighted")
print("\nF1 Score (Unique Words):", f1_unique)
print("\nReporte de clasificación:")
print(classification_report(y_test, y_pred_unique))


Vocabulario positivo size: 64553
Vocabulario negativo size: 320038

Primeras 5 predicciones: ['NEGATIVE', 'NEGATIVE', 'NEGATIVE', 'NEGATIVE', 'NEGATIVE']
Primeras 5 etiquetas reales: ['NEGATIVE' 'POSITIVE' 'NEGATIVE' 'NEGATIVE' 'NEGATIVE']

F1 Score (Unique Words): 0.8531139244010166

Reporte de clasificación:
              precision    recall  f1-score   support

    NEGATIVE       0.87      1.00      0.93     67604
    POSITIVE       1.00      0.31      0.47     14067

    accuracy                           0.88     81671
   macro avg       0.94      0.65      0.70     81671
weighted avg       0.90      0.88      0.85     81671



In [7]:
# Función principal para análisis de sentimiento
def sentiment_analysis(text: str) -> str:
    """
    Realiza análisis de sentimiento basado en vocabularios únicos.

    Args:
        text (str): Texto para analizar.

    Returns:
        str: Predicción del sentimiento ("POSITIVE" o "NEGATIVE").
    """
    return unique_word_based_sentiment(text)

#### Método basado en Word embeddings

In [8]:
import torch
from transformers import DistilBertTokenizer, DistilBertModel
from sklearn.ensemble import RandomForestClassifier
from tqdm import tqdm
import numpy as np

# 1. Cargar DistilBERT
model_name = "distilbert-base-uncased"
tokenizer = DistilBertTokenizer.from_pretrained(model_name)
model = DistilBertModel.from_pretrained(model_name)


def text_to_embedding(text, tokenizer, model):
    # Tokenizar el texto
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512)
    
    # Generar embeddings sin calcular los gradientes
    with torch.no_grad():
        outputs = model(**inputs)
    
    # Promediar las representaciones de los tokens
    return outputs.last_hidden_state.mean(dim=1).squeeze().detach().numpy()


# 2. Se toma una muestra del 5% de los datos
sample_data = data.sample(frac=0.05, random_state=42)

# 3. Dividir en entrenamiento y prueba
X_train_text, X_test_text, y_train, y_test = train_test_split(
    sample_data["clean_post"], 
    sample_data["sentiment"], 
    test_size=0.3, 
    random_state=42)

# 4. Generar embeddings con DistilBERT
X_train_bert = np.array([text_to_embedding(text, tokenizer, model) for text in tqdm(X_train_text, desc="Generating BERT train embeddings")])
X_test_bert = np.array([text_to_embedding(text, tokenizer, model) for text in tqdm(X_test_text, desc="Generating BERT test embeddings")])

# 5. Entrenar y evaluar modelo con DistilBERT
classifier_bert = RandomForestClassifier(random_state=42, n_estimators=50, max_depth=50, n_jobs=-1, min_samples_leaf=1, min_samples_split=2, max_features="sqrt")
classifier_bert.fit(X_train_bert, y_train)

y_pred_bert = classifier_bert.predict(X_test_bert)
f1_bert = f1_score(y_test, y_pred_bert, average="weighted")
print("F1 Score (DistilBERT):", f1_bert)

Generating BERT test embeddings: 100%|██████████| 4084/4084 [02:21<00:00, 28.89it/s]


F1 Score (DistilBERT): 0.7881851550269867
